# BƯỚC 1 & 2: KHẢO SÁT & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU THÔ (DATA QUALITY & PROFILING)
## DỰ ÁN: IT JOB MARKET INTELLIGENCE
---
**Mục tiêu của Notebook:**
1. **Trích xuất dữ liệu cần thiết (Feature Selection)** từ kho dữ liệu thô Bronze (bronze_listings + bronze_details_valid).
2. **Kiểm tra cấu trúc và kiểu dữ liệu (Schema & Data Types)** của toàn bộ các trường tuyển dụng.
3. **Đo lường giá trị khuyết (Missing / Null Report)**: Thống kê số lượng Null/None và chuỗi rỗng trên từng cột.
4. **Kiểm tra trùng lặp (Duplicates Report)**: Phát hiện trùng lặp theo ID nguồn, URL bài đăng và thực thể công việc [Công ty + Chức danh].
5. Đưa ra cơ sở thực tế để thực hiện bước Tiền xử lý & Làm sạch dữ liệu tiếp theo.

In [4]:
import os
import json
import pandas as pd
import numpy as np

from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [5]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 4,836 dong, 20 cot
Bang Details : 4,768 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,3096,3028
vietnamworks,1573,1573
topdev,167,167


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [6]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
print(f'  • Tỷ lệ khớp khóa (Match)  : {len(df_merged)/len(df_details)*100:.2f}% (Khớp hoàn toàn 100%)')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 4,768 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (Khớp hoàn toàn 100%)


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [7]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4768 entries, 0 to 4767
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             4768 non-null   str   
 1   source                4768 non-null   str   
 2   source_job_id         4768 non-null   str   
 3   job_url               4768 non-null   str   
 4   title_raw             4768 non-null   str   
 5   company_name_raw      4768 non-null   str   
 6   company_url           4768 non-null   str   
 7   company_size_raw      4768 non-null   str   
 8   company_location_raw  4768 non-null   str   
 9   industry_raw          4768 non-null   str   
 10  salary_raw            4768 non-null   str   
 11  experience_raw        4768 non-null   str   
 12  education_raw         4768 non-null   str   
 13  location_raw          4768 non-null   str   
 14  work_mode_raw         4768 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,4768,4768,100.00
updated_at_raw,str,0,3195,3195,67.01
overview_raw,str,0,2876,2876,60.32
company_size_raw,str,0,997,997,20.91
education_raw,str,0,762,762,15.98
working_time_raw,str,0,281,281,5.89
company_logo_url,str,0,36,36,0.76
work_mode_raw,str,0,35,35,0.73
requirements_raw,str,0,21,21,0.44
required_skills_raw,str,0,5,5,0.10


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",218,4.57


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ 39 Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **39 cột** (bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ 39 cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách 39 cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (Toàn bộ 39 cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [8]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# 39 cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(' [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,964,2223,2598
detail_id,vietnamworks_2108683,vietnamworks_2116495,topcv_2258820
source,vietnamworks,vietnamworks,topcv
source_job_id,2108683,2116495,2258820
job_url,https://www.vietnamworks.com/chuyen-gia-quan-tri-co-so-d...,https://www.vietnamworks.com/it-administrator-1791285106...,https://www.topcv.vn/viec-lam/truong-phong-ky-thuat/2258...
title_raw,Chuyên Gia Quản Trị Cơ Sở Dữ Liệu (DBA),IT Administrator,Trưởng Phòng Kỹ Thuật
company_name_raw,One Mount,Got It,CÔNG TY TNHH TƯ VẤN VÀ CÔNG NGHỆ BẢO AN
company_url,https://www.vietnamworks.com/nha-tuyen-dung/one-mount-c3...,https://www.vietnamworks.com/nha-tuyen-dung/got-it-c240117,https://www.topcv.vn/cong-ty/cong-ty-tnhh-tu-van-va-cong...
company_size_raw,1.000-4.999,100-499,10-24 nhân viên
company_location_raw,"Times City, Hanoi","9-11 Nguyễn Văn Thủ, P. Đa Kao, Quận 1, TP. Hồ Chí Minh","Số 125A, Đường Bắc Sơn, Tổ 11, Phường Phan Đình Phùng, T..."
industry_raw,Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng"


In [9]:
# ---  XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)


,source,source_job_id,job_url,crawled_at,raw_date,raw_file_name,listing_file_name
964,vietnamworks,2108683,https://www.vietnamworks.com/chuyen-gia-quan-tri-co-so-d...,2026-10-11T10:04:43.929273,2026-10-11,VNW003.json,page_003.json
2223,vietnamworks,2116495,https://www.vietnamworks.com/it-administrator-1791285106...,2026-10-12T21:04:10.640441,2026-10-12,VNW004.json,page_004.json
2598,topcv,2258820,https://www.topcv.vn/viec-lam/truong-phong-ky-thuat/2258...,2026-10-14T08:01:54.283927,2026-10-14,TCV001.json,page_001.json
3965,topdev,2141091,https://topdev.vn/viec-lam/full-stack-developer-python-f...,2026-10-27T09:15:28.962620,2026-10-27,TC001.json,page_001.json
598,topcv,1869254,https://www.topcv.vn/viec-lam/nhan-vien-san-xuat-dung-ma...,2026-10-11T20:55:17.740117,2026-10-11,TCV009.json,page_009.json
2634,topcv,2308696,https://www.topcv.vn/viec-lam/cong-tac-vien-giang-vien-c...,2026-10-14T22:06:24.643210,2026-10-14,TCV002.json,page_001.json
1056,vietnamworks,2110214,https://www.vietnamworks.com/ai-software-engineer-python...,2026-10-11T11:59:18.460175,2026-10-11,VNW004.json,page_004.json
4257,vietnamworks,2124412,https://www.vietnamworks.com/khoi-nghiep-vu-chuyen-vien-...,2026-10-28T16:13:17.703295,2026-10-28,VNW001.json,page_001.json
642,topcv,2323277,https://www.topcv.vn/viec-lam/senior-ai-scientist/232327...,2026-10-11T21:05:52.455035,2026-10-11,TCV010.json,page_009.json
2281,topcv,2280998,https://www.topcv.vn/viec-lam/truong-phong-mua-sam-cong-...,2026-10-12T21:40:39.550712,2026-10-12,TCV005.json,page_004.json



 NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)


,company_name_raw,company_url,company_logo_url,company_size_raw,company_location_raw,industry_raw,location_raw
964,One Mount,https://www.vietnamworks.com/nha-tuyen-dung/one-mount-c3...,https://images.vietnamworks.com/pictureofcompany/1a/1104...,1.000-4.999,"Times City, Hanoi",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"Times City, Hanoi"
2223,Got It,https://www.vietnamworks.com/nha-tuyen-dung/got-it-c240117,https://images.vietnamworks.com/pictureofcompany/48/1112...,100-499,"9-11 Nguyễn Văn Thủ, P. Đa Kao, Quận 1, TP. Hồ Chí Minh",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,Ho Chi Minh
2598,CÔNG TY TNHH TƯ VẤN VÀ CÔNG NGHỆ BẢO AN,https://www.topcv.vn/cong-ty/cong-ty-tnhh-tu-van-va-cong...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,10-24 nhân viên,"Số 125A, Đường Bắc Sơn, Tổ 11, Phường Phan Đình Phùng, T...","IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng","125A đường Bắc Sơn, Phường Phan Đình Phùng, Thái Nguyên"
3965,CÔNG TY TNHH ARBIN INSTRUMENTS VIỆT NAM,https://topdev.vn/nha-tuyen-dung/cong-ty-tnhh-arbin-inst...,https://assets.saramin.vn/images/2025/10/28/TopDev-logo-...,Nhân viên,"205 ÄÆ°á»ng GS1, ÄÃ´ng HÃ²a DÄ© An, Tá»nh BÃ¬nh DÆ°Æ...",Sản xuất,Hồ Chí Minh
598,CÔNG TY TNHH SẢN XUẤT THƯƠNG MẠI PIXEL DECOR,https://www.topcv.vn/cong-ty/cong-ty-tnhh-san-xuat-thuon...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,1-9 nhân viên,"270/2 Đỗ Xuân Hợp, khu phố 4, Phường Phước Long A, TP.Th...","Sản xuất, Cơ khí / Chế tạo / Tự động hóa, Công nghệ cao","Phường Phước Long, Hồ Chí Minh"
2634,CÔNG TY TNHH OPENSPACES,https://www.topcv.vn/cong-ty/cong-ty-tnhh-openspaces/205...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,10-24 nhân viên,"82 Duy Tân, Cầu Giấy, Hà Nội","IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng","Phường Ba Đình, Hà Nội"
1056,NGÂN HÀNG TMCP QUÂN ĐỘI – MBBANK,https://www.vietnamworks.com/nha-tuyen-dung/ngan-hang-tm...,https://images.vietnamworks.com/pictureofcompany/e4/1144...,,"Số 18 Lê Văn Lương, Phường Yên Hòa, Thành phố Hà Nội",Ngân hàng (Công Nghệ Thông Tin/Viễn Thông > Data Enginee...,"Số 18 Lê Văn Lương, Phường Yên Hòa, Thành phố Hà Nội"
4257,MB Ageas Life Insurance Company Limited,https://www.vietnamworks.com/nha-tuyen-dung/mb-ageas-lif...,https://images.vietnamworks.com/pictureofcompany/66/1140...,500-999,"15th Floor, MB Building - 21 Cat Linh Street, O Cho Dua ...",Bảo hiểm (Công Nghệ Thông Tin/Viễn Thông > Phân Tích Kin...,"Hà Nội, Hanoi, Vietnam"
642,FPT Software,https://www.topcv.vn/cong-ty/fpt-software/3.html,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,10000+ nhân viên,"Tòa nhà FPT Building, số 17 Duy Tân, Cầu Giấy, Hà Nội","IT phần mềm, Công nghệ thông tin","Tầng 5, Tòa nhà The Manor Crown, 62 Tố Hữu, Phường Vỹ Dạ..."
2281,CÔNG TY TÀI CHÍNH TỔNG HỢP TRÁCH NHIỆM HỮU HẠN MỘT THÀNH...,https://www.topcv.vn/cong-ty/cong-ty-tai-chinh-tong-hop-...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,1000+ nhân viên,"Tầng 12A tòa Tháp Tây, tòa nhà Lotte Center, số 54 Liễu ...","IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng","54 Liễu Giai, Phường Giảng Võ, Hà Nội"



 NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)


,title_raw,salary_raw,experience_raw,education_raw,work_mode_raw,employment_type_raw,working_time_raw,deadline_raw
964,Chuyên Gia Quản Trị Cơ Sở Dữ Liệu (DBA),60tr-100tr ₫/tháng,7 năm,,Onsite,Nhân viên,Toàn thời gian,2026-10-18T23:59:59+07:00
2223,IT Administrator,10tr-12tr ₫/tháng,1 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-11-05T23:59:59+07:00
2598,Trưởng Phòng Kỹ Thuật,15000000,3 năm,Cao Đẳng trở lên,Onsite,Trưởng/Phó phòng,Toàn thời gian,2026-10-22T23:59:59+07:00
3965,Full stack Developer (Python + Flutter/ReactJS),Negotiable MONTH,3 năm,,,Chuyên viên cấp cao,,2026-11-26
598,Nhân Viên Sản Xuất - Đứng Máy CNC,7000000,Không yêu cầu,Trung học phổ thông (Cấp 3) trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-06T23:59:59+07:00
2634,Cộng Tác Viên Giảng Viên Chuyển Đổi Số,Thoả thuận,3 năm,Đại Học trở lên,Onsite,Nhân viên,Thời vụ,2026-10-22T23:59:59+07:00
1056,AI Software Engineer (Python/ Go/ C/C++ Backend & AI Age...,20tr-50tr ₫/tháng,0 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-10-22T23:59:59+07:00
4257,[Khối Nghiệp Vụ] Chuyên Viên Cao Cấp Vận Hành Nghiệp Vụ,Thương lượng,2 năm,,Onsite,Nhân viên,Toàn thời gian,2026-11-26T23:59:59+07:00
642,Senior AI Scientist,Thoả thuận,3 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-05T23:59:59+07:00
2281,Trưởng Phòng Mua Sắm Công Nghệ Thông Tin,Thoả thuận,5 năm,Đại Học trở lên,Onsite,Trưởng/Phó phòng,Toàn thời gian,2026-10-25T23:59:59+07:00



 NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)


,skills_raw,required_skills_raw,preferred_skills_raw,posted_at_raw,published_at_raw,updated_at_raw
964,"Cơ Sở Dữ Liệu, Oracle, PostgreSQL, Python, NoSQL","Cơ Sở Dữ Liệu, Oracle, PostgreSQL, Python, NoSQL",,2026-09-18T10:27:19+07:00,2026-09-18T10:27:19+07:00,2026-09-18T10:27:19+07:00
2223,"It Infrastructure Management, Microsoft Office 365, Netw...","It Infrastructure Management, Microsoft Office 365, Netw...",,2026-10-06T18:11:46+07:00,2026-10-06T18:11:46+07:00,2026-10-06T18:11:46+07:00
2598,"3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...","3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...",,2026-09-22,2026-09-22,
3965,"Python, ReactJS, Flutter","Python, ReactJS, Flutter",,2026-10-27,2026-10-27,
598,"Không yêu cầu kinh nghiệm chuyên môn, Kỹ sư lập trình má...","Không yêu cầu kinh nghiệm chuyên môn, Kỹ sư lập trình má...",,2026-10-07,2026-10-07,
2634,"3 năm kinh nghiệm chuyên môn, Chuyên môn Công nghệ thông...","3 năm kinh nghiệm chuyên môn, Chuyên môn Công nghệ thông...",,2026-09-22,2026-09-22,
1056,"Microservice, C/C ++, PYTHON, Tensorflow, FastApi","Microservice, C/C ++, PYTHON, Tensorflow, FastApi",,2026-09-22T15:32:21+07:00,2026-09-22T15:32:21+07:00,2026-09-22T15:34:15+07:00
4257,"Bảo Hiểm, Quản Lý Tái Bảo Hiểm, Đối Soát Dữ Liệu, Thẩm Đ...","Bảo Hiểm, Quản Lý Tái Bảo Hiểm, Đối Soát Dữ Liệu, Thẩm Đ...",,2026-10-27T17:34:22+07:00,2026-10-27T17:34:22+07:00,2026-10-27T17:35:12+07:00
642,"3 năm kinh nghiệm chuyên môn, Xem tất cả việc làm AI Eng...","3 năm kinh nghiệm chuyên môn, Xem tất cả việc làm AI Eng...",,2026-10-06,2026-10-06,
2281,"2 năm kinh nghiệm quản lý, 5 năm kinh nghiệm chuyên môn,...","2 năm kinh nghiệm quản lý, 5 năm kinh nghiệm chuyên môn,...",,2026-09-25,2026-09-25,



 NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)


,title_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw
964,Chuyên Gia Quản Trị Cơ Sở Dữ Liệu (DBA),"September 19, 2019 - One Mount was established with the ...",OVERVIEW\nVị trí Chuyên viên cao cấp/ Chuyên gia Quản tr...,REQUIREMENTS:\n• Tốt nghiệp Đại học chuyên ngành Công ng...,"Thưởng: Thu nhập cạnh tranh lên tới 60-100tr/ tháng, gồm..."
2223,IT Administrator,Dayone is the parent company of Got It (2015) and Scan I...,"Manage and support IT infrastructure: Laptop, Desktop, P...",Bachelor’s degree in IT or related field.\n1 year of exp...,Chăm sóc sức khoẻ: <p>Comprehensive healthcare with Bao ...
2598,Trưởng Phòng Kỹ Thuật,,"Quản lý, điều phối và phân công công việc cho đội ngũ kỹ...",Tốt nghiệp Cao đẳng/Đại học chuyên ngành Công nghệ thông...,Thu nhập hấp dẫn theo năng lực (15–30 triệu đồng/tháng h...
3965,Full stack Developer (Python + Flutter/ReactJS),$3e,Build and maintain modern web and mobile apps for real-w...,Must Have\n3+ years in backend development using Python ...,Work with real AI engineers and hands-on projects that u...
598,Nhân Viên Sản Xuất - Đứng Máy CNC,,- Thực hiện công việc đứng máy cắt CNC\n- Đảm bảo chất l...,"Không yêu cầu kinh nghiệm, sẽ được đào tạo\nNhanh nhẹn, ...",Lương cứng: 7 – 12 triệu (Thỏa thuận tùy theo năng lực l...
2634,Cộng Tác Viên Giảng Viên Chuyển Đổi Số,,I. GIỚI THIỆU CHƯƠNG TRÌNH\nCông ty TNHH Openspaces triể...,1. Yêu cầu bắt buộc\nTốt nghiệp đại học trở lên chuyên n...,HỖ TRỢ TỪ CÔNG TY\nBộ học liệu hoàn chỉnh: slide bài giả...
1056,AI Software Engineer (Python/ Go/ C/C++ Backend & AI Age...,"Về chúng tôi:\r\n- Thành lập từ năm 1994, mô hình tập đo...",Xây dựng phát triển các sản phẩm AI có khả năng tự động ...,- Tốt nghiệp Đại học các chuyên ngành: Công nghệ thông t...,"Thưởng: MBers còn nhận quà sinh nhật ý nghĩa, nghỉ nguyê..."
4257,[Khối Nghiệp Vụ] Chuyên Viên Cao Cấp Vận Hành Nghiệp Vụ,"Established in 2016, MB Life is a joint venture of Milit...",1. Thực hiện đối soát dữ liệu vận hành với các bên thứ b...,"• Tốt nghiệp đại học chuyên ngành Toán, Khoa học dữ liệu...","Thưởng: Cơ cấu thu nhập hấp dẫn: Lương cứng, Phụ cấp cố ..."
642,Senior AI Scientist,,Optimize an existing AI Chatbot system for the client.\n...,Experience in Python Backend and AI system development.\...,"Signing bonus for Senior level up\nInternational, profes..."
2281,Trưởng Phòng Mua Sắm Công Nghệ Thông Tin,,1. Triển khai hoạt động mua sắm CNTT\n• Thực hiện và giá...,1. Trình độ học vấn: Tốt nghiệp Đại học chuyên ngành Ki...,- Thời gian làm việc: 5 ngày/tuần (Thứ 2 đến thứ 6)\n- M...


In [10]:
# ---  BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ 39 CỘT (SAMPLE 10 DÒNG) ---
print(' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):


,detail_id,source,source_job_id,job_url,title_raw,company_name_raw,company_url,company_size_raw,company_location_raw,industry_raw,salary_raw,experience_raw,education_raw,location_raw,work_mode_raw,employment_type_raw,working_time_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw,required_skills_raw,preferred_skills_raw,published_at_raw,updated_at_raw,deadline_raw,crawled_at,raw_file_name,raw_date,dq_status,dq_errors,dq_checked_at,_dq_status,_dq_errors,_dq_checked_at,company_logo_url,skills_raw,posted_at_raw,listing_file_name
964,vietnamworks_2108683,vietnamworks,2108683,https://www.vietnamworks.com/chuyen-gia-quan-tri-co-so-d...,Chuyên Gia Quản Trị Cơ Sở Dữ Liệu (DBA),One Mount,https://www.vietnamworks.com/nha-tuyen-dung/one-mount-c3...,1.000-4.999,"Times City, Hanoi",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,60tr-100tr ₫/tháng,7 năm,,"Times City, Hanoi",Onsite,Nhân viên,Toàn thời gian,"September 19, 2019 - One Mount was established with the ...",OVERVIEW\nVị trí Chuyên viên cao cấp/ Chuyên gia Quản tr...,REQUIREMENTS:\n• Tốt nghiệp Đại học chuyên ngành Công ng...,"Thưởng: Thu nhập cạnh tranh lên tới 60-100tr/ tháng, gồm...","Cơ Sở Dữ Liệu, Oracle, PostgreSQL, Python, NoSQL",,2026-09-18T10:27:19+07:00,2026-09-18T10:27:19+07:00,2026-10-18T23:59:59+07:00,2026-10-11T10:04:43.929273,VNW003.json,2026-10-11,valid,[],2026-10-30T17:31:07.668061,valid,[],2026-10-30T17:31:07.668061,https://images.vietnamworks.com/pictureofcompany/1a/1104...,"Cơ Sở Dữ Liệu, Oracle, PostgreSQL, Python, NoSQL",2026-09-18T10:27:19+07:00,page_003.json
2223,vietnamworks_2116495,vietnamworks,2116495,https://www.vietnamworks.com/it-administrator-1791285106...,IT Administrator,Got It,https://www.vietnamworks.com/nha-tuyen-dung/got-it-c240117,100-499,"9-11 Nguyễn Văn Thủ, P. Đa Kao, Quận 1, TP. Hồ Chí Minh",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,10tr-12tr ₫/tháng,1 năm,Cử nhân,Ho Chi Minh,Onsite,Nhân viên,Toàn thời gian,Dayone is the parent company of Got It (2015) and Scan I...,"Manage and support IT infrastructure: Laptop, Desktop, P...",Bachelor’s degree in IT or related field.\n1 year of exp...,Chăm sóc sức khoẻ: <p>Comprehensive healthcare with Bao ...,"It Infrastructure Management, Microsoft Office 365, Netw...",,2026-10-06T18:11:46+07:00,2026-10-06T18:11:46+07:00,2026-11-05T23:59:59+07:00,2026-10-12T21:04:10.640441,VNW004.json,2026-10-12,valid,[],2026-10-30T17:31:07.678807,valid,[],2026-10-30T17:31:07.678807,https://images.vietnamworks.com/pictureofcompany/48/1112...,"It Infrastructure Management, Microsoft Office 365, Netw...",2026-10-06T18:11:46+07:00,page_004.json
2598,topcv_2258820,topcv,2258820,https://www.topcv.vn/viec-lam/truong-phong-ky-thuat/2258...,Trưởng Phòng Kỹ Thuật,CÔNG TY TNHH TƯ VẤN VÀ CÔNG NGHỆ BẢO AN,https://www.topcv.vn/cong-ty/cong-ty-tnhh-tu-van-va-cong...,10-24 nhân viên,"Số 125A, Đường Bắc Sơn, Tổ 11, Phường Phan Đình Phùng, T...","IT phần mềm, Công nghệ thông tin, IT Phần cứng / Mạng",15000000,3 năm,Cao Đẳng trở lên,"125A đường Bắc Sơn, Phường Phan Đình Phùng, Thái Nguyên",Onsite,Trưởng/Phó phòng,Toàn thời gian,,"Quản lý, điều phối và phân công công việc cho đội ngũ kỹ...",Tốt nghiệp Cao đẳng/Đại học chuyên ngành Công nghệ thông...,Thu nhập hấp dẫn theo năng lực (15–30 triệu đồng/tháng h...,"3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...",,2026-09-22,,2026-10-22T23:59:59+07:00,2026-10-14T08:01:54.283927,TCV001.json,2026-10-14,valid,[],2026-10-30T17:31:07.072764,valid,[],2026-10-30T17:31:07.072764,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...",2026-09-22,page_001.json
3965,topdev_2141091,topdev,2141091,https://topdev.vn/viec-lam/full-stack-developer-python-f...,Full stack Developer (Python + Flutter/ReactJS),CÔNG TY TNHH ARBIN INSTRUMENTS VIỆT NAM,https://topdev.vn/nha-tuyen-dung/cong-ty-tnhh-arbin-inst...,Nhân viên,"205 ÄÆ°á»ng GS1, ÄÃ´ng HÃ²a DÄ© An, Tá»nh BÃ¬nh DÆ°Æ...",Sản xuất,Nego